In [ ]:
import os
import warnings
import pickle

from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, adjusted_mutual_info_score, balanced_accuracy_score
from sklearn.utils import shuffle

from catboost import CatBoostClassifier

from cstarters.ml_utils import avg, load_dataset, featurize_alignment, confusion_matrix_plot, add_stratify_column, drop_labels, average_value_feature, plot_feature_importances, stdev_accuracy

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*probability.*deprecated.*"
)

random_seed = 0

In [ ]:
data_dir = "../../data"
out_dir = "../../out"

In [ ]:
df = load_dataset(path = os.path.join(data_dir, "cstarters_dataset.csv"))

aa_properties = {
    'WOLS870101': [96, 97, 98, 320, 338, 340, 342, 349, 351, 352, 356, 357, 814, 818, 902, 924, 932, 936, 938, 707, 602],
    'WOLS870102': [96, 97, 98, 320, 338, 340, 342, 349, 351, 352, 356, 357, 814, 818, 902, 924, 932, 936, 938, 707, 602],
    'WOLS870103': [96, 97, 98, 320, 338, 340, 342, 349, 351, 352, 356, 357, 814, 818, 902, 924, 932, 936, 938, 707, 602],
    'GRAR740102': [96, 97, 98, 320, 338, 340, 342, 349, 351, 352, 356, 357, 814, 818, 902, 924, 932, 936, 938, 707, 602],
    'RADA880108': [96, 97, 98, 320, 338, 340, 342, 349, 351, 352, 356, 357, 814, 818, 902, 924, 932, 936, 938, 707, 602]
}

df = average_value_feature(
    dataset_df=df,
    positions= [96, 97, 98, 320, 338, 340, 342, 349, 351, 352, 356, 357, 814, 818, 902, 924, 932, 936, 938],
    aa_property="vdwvol",
    output_column="avg_size_tunnel_L",
)

df = average_value_feature(
    dataset_df=df,
    positions=[96, 97, 98, 349, 352, 814, 818, 924, 932, 936, 938, 944],
    aa_property="vdwvol",
    output_column="avg_size_tunnel_S",
)

df = featurize_alignment(df, aa_properties)
ml_df_full = df.drop(columns=["starter_beta_hydroxylation"])
ml_df_full = drop_labels(ml_df_full, 'starter_category', ['AR', 'FA'])
stratify_columns = ['starter_category', 'ncbi_organism_genus']
ml_df_full = add_stratify_column(ml_df_full, stratify_columns)
ml_df_full = ml_df_full.drop(columns=['ncbi_organism_genus'])
ml_df_train, ml_df_test = train_test_split(ml_df_full, test_size = 0.2, stratify = ml_df_full['stratify_column'], random_state = random_seed)

y_train = ml_df_train['starter_category']
X_train = ml_df_train.drop(columns=['cs_accession', 'starter_category', 'stratify_column', ])
X_train, y_train = shuffle(X_train, y_train, random_state=random_seed)

y_test = ml_df_test['starter_category']
X_test = ml_df_test.drop(columns=['cs_accession', 'starter_category', 'stratify_column'])
X_test, y_test = shuffle(X_test, y_test, random_state=random_seed)

y_full = ml_df_full['starter_category']
X_full = ml_df_full.drop(columns=['cs_accession', 'starter_category', 'stratify_column'])
X_full, y_full = shuffle(X_full, y_full, random_state=random_seed)


# Train

In [ ]:
# RF train
rf_params = dict(
    class_weight='balanced',
    ccp_alpha=0.01,
    max_depth=15,
    min_samples_leaf=1,
    max_leaf_nodes=4,
    n_estimators=100,
    min_samples_split=8,
    max_features="log2",
    random_state=random_seed
)

model = RandomForestClassifier(**rf_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • RF • Train Dataset")

In [ ]:
# SVM train
svm_params = dict(
    probability=True,
    C = 0.1, 
    kernel='linear',
    degree=3,
    class_weight='balanced',
    random_state=random_seed
)
model = svm.SVC(**svm_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • SVM • Train Dataset")

In [ ]:
# Logistic Regression train
lr_params = dict(
    solver='newton-cg', 
    class_weight='balanced', 
    # penalty='l2', 
    C=0.01
)
model = LogisticRegression(**lr_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • LR • Train Dataset")

In [ ]:
# catboost train
cat_params = dict(
    iterations=175,
    learning_rate=0.2,
    depth=4,
    l2_leaf_reg=3,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=random_seed,
    verbose=0
)

model = CatBoostClassifier(**cat_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, "avg:", avg(cv))

cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["SCFA", "MCFA", "LCFA"]),["SCFA", "MCFA", "LCFA"], title="Length • CatBoost • Train Dataset")

# Test

In [ ]:
# RF test
model = RandomForestClassifier(**rf_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • RF • Test Dataset")
plot_feature_importances(model, X_test.columns)

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))


In [ ]:
# SVM test
model = svm.SVC(**svm_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • SVM • Test Dataset")

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

In [ ]:
# Logistic Regression test
model = LogisticRegression(**lr_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • LR • Test Dataset")

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))


In [ ]:
# catboost test
model = CatBoostClassifier(**cat_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length • CatBoost • Test Dataset")

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred.ravel()), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred.ravel()), 3))

# Full

In [ ]:
# RF full


cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"])

stdev_accuracy(model, X_full, y_full, iterations=100)

In [ ]:
# SVM full
model = svm.SVC(**svm_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"])
stdev_accuracy(model, X_full, y_full, iterations=100)

In [ ]:
# Logistic Regression full
model = LogisticRegression(**lr_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"])

In [ ]:
# catboost full
model = CatBoostClassifier(**cat_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["SCFA", "MCFA", "LCFA"]), ["SCFA", "MCFA", "LCFA"], title="Length Classification • CatBoost • Full Dataset")

stdev_accuracy(model, X_full, y_full, iterations=100)


In [ ]:
model = CatBoostClassifier(**cat_params)

model.fit(X_full, y_full)

pickle.dump(model, open(os.path.join(out_dir, "ml_length.pkl"), "wb"))